## 7 Эксплуатация: локальный прогноз и итоговые выводы

**Входы**

- Обученный Pipeline, зафиксированный порог и новые заявки без loan_status.

**Выходы**

- Оценка риска и класс для каждой новой заявки.
- Контракт применения, план наблюдения и окончательная таблица выводов.

**Что делаем**

Применяем обученную модель к трём искусственным заявкам с зафиксированным порогом классификации.

In [ ]:
# Только для разработки: открыть раздел можно из notebooks/ или корня проекта
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### Контракт применения

Вход — DataFrame ровно с 11 исходными признаками; единицы те же, что в датасете. Порядок колонок восстанавливается по схеме. Пропуски допускаются и обрабатываются обученным Pipeline. Неизвестный грейд вне A–G, неверный набор колонок, бесконечности и явно недопустимые числовые значения отклоняются. Недопустимый стаж заменяется пропуском по принятому правилу. Новые номинальные категории не прерывают прогноз, но требуют регистрации и анализа.

Выход — модельная оценка риска и бинарный класс по сохранённому порогу; порядок и число строк сохраняются. Это демонстрация механики, истинных исходов этих искусственных заявок нет. Числа predict_proba не гарантируют точную индивидуальную вероятность без отдельной проверки калибровки.

In [ ]:
exec(compile(__import__('pathlib').Path('src/notebooks/_code/inference.py').read_text(encoding='utf-8'), 'src/notebooks/_code/inference.py', 'exec'), globals())

#### Наблюдение после допуска к реальному применению

| Что наблюдать | Как реагировать |
| --- | --- |
| Схема, диапазоны, единицы, пропуски, новые категории | Логировать нарушения; ошибочную заявку возвращать на исправление |
| Сдвиг распределений относительно train, включая грейд и ставку | Исследовать источник и временные срезы; не менять порог автоматически |
| AP, recall/precision, Brier и калибровка при поступлении фактических исходов | Оценивать с учётом задержки дефолта и размеров групп |
| Стоимость ошибок и качество по значимым группам | Определить допустимые уровни с предметными специалистами до внедрения |
| Версии данных, библиотек, Pipeline и порога | Хранить совместно; новое обучение проверять на новом отложенном периоде |

Таблица пропусков иллюстрирует контроль входных данных. Трёх искусственных заявок недостаточно для оценки сдвига распределений или качества прогнозов на новых данных.